# Q52: Feature Selection Strategies
**Topic:** Classical-ml | **Difficulty:** Medium | **Time:** 12 min
**Sheet:** Grind75ML

---

## Question
How to handle feature selection? What methods do you use?

## Detailed Answer

### Three Categories of Feature Selection

#### 1. Filter Methods (Statistical)
Rank features by statistical measures independent of the model:
- **Correlation** with target (Pearson for continuous, point-biserial for binary)
- **Mutual Information**: Captures non-linear relationships
- **Chi-squared test**: For categorical features
- **Variance threshold**: Remove near-zero variance features
- **ANOVA F-test**: For continuous features + categorical target
- **Pros**: Fast, model-agnostic. **Cons**: Ignores feature interactions

#### 2. Wrapper Methods
Use model performance to evaluate feature subsets:
- **Forward selection**: Start empty, add best feature iteratively
- **Backward elimination**: Start with all, remove worst iteratively
- **Recursive Feature Elimination (RFE)**: Train model, remove least important features, repeat
- **Pros**: Considers feature interactions. **Cons**: Computationally expensive ($O(2^d)$ subsets)

#### 3. Embedded Methods
Feature selection is part of model training:
- **L1 regularization (Lasso)**: Drives coefficients to exactly zero → automatic selection
- **Tree-based importance**: Random Forest / XGBoost feature importance
- **Attention weights**: In Transformers, attention reveals important features
- **Pros**: Best balance of speed and quality. **Cons**: Model-specific

### Recommended Pipeline
```
1. Remove near-zero variance features (filter)
2. Remove highly correlated features (r > 0.95)
3. Train XGBoost, use feature importance (embedded)
4. Optionally: RFE with final model (wrapper)
```

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.feature_selection import mutual_info_classif, SelectKBest, RFE
from sklearn.ensemble import RandomForestClassifier

X, y = make_classification(n_samples=1000, n_features=20, n_informative=5, n_redundant=5, random_state=42)

# Filter: Mutual Information
mi = mutual_info_classif(X, y, random_state=42)
top_mi = np.argsort(mi)[::-1][:5]
print(f'Top 5 features by MI: {top_mi}')

# Embedded: RF importance
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X, y)
top_rf = np.argsort(rf.feature_importances_)[::-1][:5]
print(f'Top 5 features by RF: {top_rf}')

# Wrapper: RFE
rfe = RFE(RandomForestClassifier(n_estimators=50, random_state=42), n_features_to_select=5)
rfe.fit(X, y)
print(f'Top 5 by RFE: {np.where(rfe.support_)[0]}')

## Key Takeaways
- Start with **filter methods** (fast), refine with **embedded methods** (RF/XGBoost importance)
- **L1 regularization** is a clean embedded approach for linear models
- **Tree feature importance** is the industry standard for tabular data
- Always validate: compare model performance with vs. without selected features